In [10]:
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import csr_matrix
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zipfile
# import urllib.request
import sys

print("Current Working Directory: ", os.getcwd())

Current Working Directory:  C:\Users\Connor\Documents\3. School\Projects\CPTS\Project\code\315SongRecomend


In [11]:
data = pd.read_csv('./data/used/train_triplets.txt', sep='\t', header=None, names=['user_id', 'song_id', 'play_count'])

# Add new columns by searching unique_tracks.txt to get the track_name and artist_name for each song_id
# unique_tracks.txt: track_id<SEP>song_id<SEP>artist_name<SEP>track_name

unique_tracks = pd.read_csv('./data/used/unique_tracks.txt', sep='<SEP>', header=None, names=['track_id', 'song_id', 'artist_name', 'track_name'])
data = pd.merge(data, unique_tracks, on='song_id', how='left')

# save data to csv
data.to_csv('./data/used/output/merged_data.csv', index=False)


C:\Users\Connor\AppData\Local\Temp\ipykernel_28456\2802193819.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  unique_tracks = pd.read_csv('./data/used/unique_tracks.txt', sep='<SEP>', header=None, names=['track_id', 'song_id', 'artist_name', 'track_name'])


In [6]:
# Helper functions

# Get the top N songs for a given user_id
def get_top_songs_for_user(user_id, n=10):
    user_data = data[data['user_id'] == user_id]
    top_songs = user_data.groupby('song_id')['play_count'].sum().sort_values(ascending=False).head(n)
    top_songs = top_songs.reset_index()
    top_songs = pd.merge(top_songs, unique_tracks[['song_id', 'track_name', 'artist_name']], on='song_id', how='left')
    return top_songs[['song_id', 'track_name', 'artist_name', 'play_count']]

# Get the top N artists for a given user_id
def get_top_artists_for_user(user_id, n=10):
    user_data = data[data['user_id'] == user_id]
    top_artists = user_data.groupby('artist_name')['play_count'].sum().sort_values(ascending=False).head(n)
    top_artists = top_artists.reset_index()
    return top_artists[['artist_name', 'play_count']]

# Get a song id given a track name (loose match)
def get_song_id_from_track_name(track_name):
    track_name = track_name.lower()
    unique_tracks['track_name_lower'] = unique_tracks['track_name'].str.lower()
    match = unique_tracks[unique_tracks['track_name_lower'].str.contains(track_name, na=False)]
    if not match.empty:
        return match.iloc[0]['song_id']
    else:
        return None

# Get list of n top songs by a given artist name (loose match)
def get_top_songs_by_artist(artist_name, n=10):
    artist_name = artist_name.lower()
    unique_tracks['artist_name_lower'] = unique_tracks['artist_name'].str.lower()
    match = unique_tracks[unique_tracks['artist_name_lower'].str.contains(artist_name, na=False)]
    if not match.empty:
        top_songs = match.groupby('song_id')['track_name'].first().head(n).reset_index()
        return top_songs[['song_id', 'track_name']]
    else:
        return None



In [ ]:
# Find top 20 most played songs and display their song_id, track_name, artist_name, and total play_count
top_songs = data.groupby('song_id')['play_count'].sum().sort_values(ascending=False).head(20)
top_songs = top_songs.reset_index()
top_songs = pd.merge(top_songs, unique_tracks[['song_id', 'track_name', 'artist_name']], on='song_id', how='left')
# print and Save output to csv
# print(top_songs[['song_id', 'track_name', 'artist_name', 'play_count']])
top_songs.to_csv('./data/used/output/top_20_songs.csv', index=False)

In [ ]:
# Find the top 20 most played artists and display their artist_name and total play_count
top_artists = data.groupby('artist_name')['play_count'].sum().sort_values(ascending=False).head(20)
top_artists = top_artists.reset_index()
# Print and save
# print(top_artists[['artist_name', 'play_count']])
top_artists.to_csv('./data/used/output/top_20_artists.csv', index=False)